In [2]:
#Level 1Task 1- Data cleaning and Preprocessing

In [3]:
#Import libraries 
import pandas as pd 
import numpy as np 

In [4]:
#Load dataset "Stock Prices Data Set", check how many rows and columns it contains, show the first five rows in the dataset.
df= pd.read_csv("Stock Prices Data Set.csv")

print ("Shape:", df.shape)
df.head()

Shape: (497472, 7)


,symbol,date,open,high,low,close,volume
0,AAL,2014-01-02,25.0700,25.8200,25.0600,25.3600,8998943
1,AAPL,2014-01-02,79.3828,79.5756,78.8601,79.0185,58791957
2,AAP,2014-01-02,110.3600,111.8800,109.2900,109.7400,542711
3,ABBV,2014-01-02,52.1200,52.3300,51.5200,51.9800,4569061
4,ABC,2014-01-02,70.1100,70.2300,69.4800,69.8900,1148391


In [5]:
#This shows what the dataset "Stock Prices Data Set" contains
print("Dataset Info")
df.info()

Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 497472 entries, 0 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   symbol  497472 non-null  object 
 1   date    497472 non-null  object 
 2   open    497461 non-null  float64
 3   high    497464 non-null  float64
 4   low     497464 non-null  float64
 5   close   497472 non-null  float64
 6   volume  497472 non-null  int64  
dtypes: float64(4), int64(1), object(2)
memory usage: 26.6+ MB


In [6]:
#Check for missing values 
#This checks how many missing values are in each column of the dataset 
print("Missing Values")
print(df.isnull().sum())

Missing Values
symbol     0
date       0
open      11
high       8
low        8
close      0
volume     0
dtype: int64


In [7]:
#Drop rows with missing open/high/low values
df_cleaned = df.dropna(subset=['open', 'high', 'low']).copy()

# Confirm the rows are gone
print("Shape before cleaning:", df.shape)
print("Shape after cleaning:", df_cleaned.shape)
print("\nMissing values after cleaning:")
print(df_cleaned.isnull().sum())

Shape before cleaning: (497472, 7)
Shape after cleaning: (497461, 7)

Missing values after cleaning:
symbol    0
date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64


In [8]:
#Check for any duplicate rows
print("Duplicates:", df.duplicated().sum())

Duplicates: 0


In [9]:
# Convert 'date' column from text to datetime
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])

#Standardize "symbol" column: remove extra whitespace, ensure consistent case
df_cleaned["symbol"] = df_cleaned["symbol"].str.strip().str.upper()

# Confirm it worked
print(df_cleaned.dtypes)
print("\nSample of cleaned data: ")
print(df_cleaned.head())

print("\nDate range in dataset:")
print("Earliest:", df_cleaned['date'].min())
print("Latest:", df_cleaned['date'].max())

print("Unique symbols:", df_cleaned["symbol"].nunique())

symbol            object
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object

Sample of cleaned data: 
  symbol       date      open      high       low     close    volume
0    AAL 2014-01-02   25.0700   25.8200   25.0600   25.3600   8998943
1   AAPL 2014-01-02   79.3828   79.5756   78.8601   79.0185  58791957
2    AAP 2014-01-02  110.3600  111.8800  109.2900  109.7400    542711
3   ABBV 2014-01-02   52.1200   52.3300   51.5200   51.9800   4569061
4    ABC 2014-01-02   70.1100   70.2300   69.4800   69.8900   1148391

Date range in dataset:
Earliest: 2014-01-02 00:00:00
Latest: 2017-12-29 00:00:00
Unique symbols: 505


In [11]:
#Save the cleaned dataset
df_cleaned.to_csv("Stock_Prices_Cleaned.csv", index=False)

print("Dataset saved sucessfully as 'Stock_Prices_Cleaned.csv'")
print("Shape of saved file:", df_cleaned.shape)

Dataset saved sucessfully as 'Stock_Prices_Cleaned.csv'
Shape of saved file: (497461, 7)


In [ ]:
#Level 1 Task 3- Basic Data Visualization

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
#Line Chart
aapl = df_cleaned[df_cleaned['symbol'] == 'AAPL'].sort_values('date')

plt.figure(figsize=(12, 5))
plt.plot(aapl['date'], aapl['close'], color='steelblue')
plt.title('AAPL Closing Price Over Time (2014-2017)')
plt.xlabel('Date')
plt.ylabel('Closing Price ($)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Barplot
symbols_to_compare = ['AAPL', 'AAL', 'ABT', 'ADBE', 'ADI']
avg_close = df_cleaned[df_cleaned['symbol'].isin(symbols_to_compare)].groupby('symbol')['close'].mean()

plt.figure(figsize=(8, 5))
avg_close.plot(kind='bar', color='seagreen')
plt.title('Average Closing Price by Stock (2014-2017)')
plt.xlabel('Stock Symbol')
plt.ylabel('Average Closing Price ($)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(aapl['volume'], aapl['close'], alpha=0.5, color='darkorange')
plt.title('AAPL: Trading Volume vs Closing Price')
plt.xlabel('Volume')
plt.ylabel('Closing Price ($)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Multi-line chart with a legend
plt.figure(figsize=(12, 6))

for symbol in ['AAPL', 'ABT', 'ADBE']:
    stock_data = df_cleaned[df_cleaned['symbol'] == symbol].sort_values('date')
    plt.plot(stock_data['date'], stock_data['close'], label=symbol)

plt.title('Closing Price Comparison: AAPL vs ABT vs ADBE (2014-2017)')
plt.xlabel('Date')
plt.ylabel('Closing Price ($)')
plt.legend(title='Stock Symbol')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Level 2 Task 1- Regression Analysis

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
#Split into training and testing sets
# Define feature (X) and target (y)
X = aapl[['open']]
y = aapl['close']

# Split into train (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)
print("Intercept:", model.intercept_)
print("Coefficient (open price):", model.coef_[0])

# Interpretation
print(f"\nFor every $1 increase in 'open' price, 'close' price changes by ${model.coef_[0]:.4f}")
print(f"When 'open' price is $0, the predicted 'close' price is ${model.intercept_:.2f}")

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate using R-squared and Mean Squared Error
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("R-squared (R²):", r2)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)

In [ ]:
# Visualise the regression fit 
plt.figure(figsize=(8, 6))
plt.scatter(X_test, y_test, alpha=0.5, label='Actual', color='steelblue')
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Predicted (regression line)')
plt.title('AAPL: Open Price vs Close Price (Regression)')
plt.xlabel('Open Price ($)')
plt.ylabel('Close Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Level 2 Task 2- Time Series Analysis 

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
# Focus on one stock, sorted by date
aapl = df_cleaned[df_cleaned['symbol'] == 'AAPL'].sort_values('date')

# Set date as the index (required for time-series functions)
aapl = aapl.set_index('date')

print("Shape:", aapl.shape)
print(aapl[['close']].head())

In [ ]:
#Plot the time series and identify patterns 
plt.figure(figsize=(12, 5))
plt.plot(aapl.index, aapl['close'], color='steelblue')
plt.title('AAPL Closing Price (2014-2017)')
plt.xlabel('Date')
plt.ylabel('Closing Price ($)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Seasonal decomposition (period=252 = approx. trading days in a year)
decomposition = seasonal_decompose(aapl['close'], model='additive', period=252)

fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate 30-day and 90-day moving averages
aapl['MA_30'] = aapl['close'].rolling(window=30).mean()
aapl['MA_90'] = aapl['close'].rolling(window=90).mean()

plt.figure(figsize=(12, 6))
plt.plot(aapl.index, aapl['close'], label='Actual Close', alpha=0.4, color='gray')
plt.plot(aapl.index, aapl['MA_30'], label='30-Day Moving Average', color='blue')
plt.plot(aapl.index, aapl['MA_90'], label='90-Day Moving Average', color='red')
plt.title('AAPL Closing Price with Moving Averages')
plt.xlabel('Date')
plt.ylabel('Closing Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Level 3 Task 1- Predictive Modeling(Classification)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
aapl['next_close'] = aapl['close'].shift(-1)
aapl['target'] = (aapl['next_close'] > aapl['close']).astype(int)
aapl = aapl.dropna(subset=['next_close'])

In [ ]:
# Features: today's open, high, low, close, volume
X = aapl[['open', 'high', 'low', 'close', 'volume']]
y = aapl['target']

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set:", X_train.shape[0], "rows")
print("Testing set:", X_test.shape[0], "rows")

In [ ]:
# Logistic Regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

# Decision Tree
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
#Evaluate each model
def evaluate(y_test, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))
    print()

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_tree, "Decision Tree")
evaluate(y_test, y_pred_rf, "Random Forest")


In [ ]:
#Hyperparameter tuning with GridSearchCV (on Random Forest)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='f1')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best F1 score (cross-validated):", grid_search.best_score_)

# Evaluate the tuned model
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
evaluate(y_test, y_pred_best, "Tuned Random Forest")